# Romania 2001: four-county descriptive placement pilot

This notebook lets Charles run the **Alba, Caraș-Severin, Galați, and Tulcea** comparison in Cursor. The saved Romanian Ministry webpages stay in the private Desktop cache. The Python companion writes only **name-free summaries, program lists, and figures** to a new, timestamped output folder.

We are asking a descriptive question: among applicants with similar national-exam scores, how does observed placement in designated selective programs vary with the exam performance of their *other observed applicants from the same originating gymnasium*? This does **not** establish a causal peer or congestion effect. The observed success fraction is **not** institutional \(K/N\), because we do not have each student's program preferences.

The source and design checks are already recorded in:
- [Four-county source reconciliation](../outputs/romania_2001_four_county_source_pilot/full_gymnasium_reconciliation.md)
- [Program-identity source gate](../outputs/romania_2001_four_county_source_pilot/program_identity_source_gate.md)
- [Four-county decision log](../docs/decisions/EDUCATION_20261002_Romania_four_county_pilot_design_decisions.md)

**Run order:** Cell 1 settings → Cell 2 read-only source preview → Cell 3 researcher-controlled analysis → Cell 4 display the saved report and a chosen figure. Select the `sports_net` kernel in Cursor before running.

In [ ]:
# CELL 1 — VISIBLE RESEARCH SETTINGS
# This cell locates the repo from Cursor's current working directory. It imports
# the companion Python code, but importing that file DOES NOT fetch or analyze.
from pathlib import Path
import importlib
import sys
import pandas as pd
from IPython.display import display, Markdown, Image

def find_analysis_root():
    for parent in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        candidate = parent / "3-Master_Plan/VECTOR_work/new_VECTOR_work/education_analysis"
        if candidate.is_dir():
            return candidate
        if parent.name == "education_analysis" and (parent / "code").is_dir():
            return parent
    raise FileNotFoundError("Open this notebook from inside the dissertation repository.")

ANALYSIS_ROOT = find_analysis_root()
CODE_DIR = ANALYSIS_ROOT / "code"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
import EDUCATION_20261004_romania_four_county_pilot as pilot
importlib.reload(pilot)  # Picks up a corrected .py file without restarting Cursor.

# Primary definition agreed October 2–4: one highest cutoff tier, full programs,
# six original subjects, vocational destinations included. Tier 2 is a planned
# cumulative comparison, not a replacement chosen after looking at the answer.
TOP_TIERS = (1, 2)
REQUIRE_FULL_PROGRAMS = True
COMBINE_PROGRAM_CATEGORIES = False
INCLUDE_VOCATIONAL_IN_PRIMARY = True

# Separate, named comparisons. They do not change the primary definition.
COMPARE_EXCLUDING_AMBIGUOUS_EXACT_PROGRAMS = True
COMPARE_EXCLUDING_VOCATIONAL_DESTINATIONS = True
COMPARE_POOLED_SCORE_BANDS = True

# Three percentile boundaries, ordered HIGH to LOW. Default (90, 75, 50)
# makes the red band top 10%, orange excellence band 75th–90th, gray
# 50th–75th, and blue below 50th. For example, (95, 80, 50) makes
# orange 80th–95th and red top 5%. These are OWN exam-score percentiles.
# Re-run Cell 1 and then Cell 3 after changing this tuple.
AI_PERCENTILE_CUTS = (90, 75, 50)

# Plot resolution only. These settings do not alter placement classifications.
PEER_STRENGTH_BINS = 8
HERO_BINS = 16

settings = pilot.Settings(
    top_tiers=TOP_TIERS,
    require_full_programs=REQUIRE_FULL_PROGRAMS,
    combine_program_categories=COMBINE_PROGRAM_CATEGORIES,
    include_vocational_in_primary=INCLUDE_VOCATIONAL_IN_PRIMARY,
    compare_excluding_ambiguous=COMPARE_EXCLUDING_AMBIGUOUS_EXACT_PROGRAMS,
    compare_excluding_vocational=COMPARE_EXCLUDING_VOCATIONAL_DESTINATIONS,
    compare_pooled_score_bands=COMPARE_POOLED_SCORE_BANDS,
    ai_percentile_cuts=AI_PERCENTILE_CUTS,
    peer_strength_bins=PEER_STRENGTH_BINS,
    hero_bins=HERO_BINS,
)
settings.validate()
print("Analysis code:", pilot.__file__)
print("Primary: top one full-program tier within six original subjects.")
print("Planned comparison: cumulative top two tiers.")
print("Vocational destinations remain in the primary denominator.")
print("Own-score percentile cuts (top, excellence start, middle):", AI_PERCENTILE_CUTS)
print("No analysis or network request has run yet.")

## Cell 2: review the saved source gate

This cell reads **county-level counts only**. “Ambiguous exact program” means the original county-wide placement row could correspond to more than one program in the same school/profile/subject. The source audit found that all 558 such placements are outside both designated top-one and top-two sets under the agreed full-program, six-subject rule. They therefore count as nonsuccesses in the primary denominator, while their exact program identities remain unresolved.

One Galați applicant has no matching saved placement or unassigned outcome. That person remains a peer but stays outside the outcome denominator; the generated summary will show a one-case bound. A student placed outside their originating county also remains a peer but is outside this pilot's **within-county** outcome denominator.

In [ ]:
# CELL 2 — READ-ONLY SOURCE PREVIEW
# No personal records are loaded here. No analysis is calculated.
gate_file = ANALYSIS_ROOT / "outputs/romania_2001_four_county_source_pilot/program_identity_source_gate.csv"
gate = pd.read_csv(gate_file)
show = gate[[
    "county", "gymnasium_applicant_rows", "one_local_placement",
    "local_program_ambiguous", "ambiguous_might_be_top1",
    "ambiguous_might_be_top2", "confirmed_unassigned",
    "placed_other_county", "unresolved_or_ambiguous_outcome",
    "no_observed_peer",
]].rename(columns={
    "county": "Origin county",
    "gymnasium_applicant_rows": "Gymnasium applicants",
    "one_local_placement": "Local placements",
    "local_program_ambiguous": "Exact program unclear",
    "ambiguous_might_be_top1": "Could be top 1",
    "ambiguous_might_be_top2": "Could be top 2",
    "confirmed_unassigned": "Unassigned",
    "placed_other_county": "Placed in another county",
    "unresolved_or_ambiguous_outcome": "Outcome unresolved",
    "no_observed_peer": "No observed peer",
})
display(show)
print("This is the saved source inventory. It is not a HERO result.")

## Cell 3: you start the analysis

Set `RUN_ANALYSIS = True` and run the next cell when you are ready. The companion will print four progress stages: verifying saved pages; setting exam-score bands; calculating county outcomes and plot cells; saving tables and figures. It makes **no Wayback requests**. A new timestamped folder preserves every prior run.

The program rule is applied separately in each county. The red and orange bands describe **own national-exam performance within the originating county**, using all recovered admission-round applicants before outcome exclusions. Cell 1 sets their percentile boundaries with `AI_PERCENTILE_CUTS`; the default gives red = top 10% and orange = 75th–90th. A separately saved comparison uses pooled four-county score bands. The horizontal axis uses the **original exam-score scale** for the other observed applicants in the gymnasium.

In [ ]:
# CELL 3 — RESEARCHER-CONTROLLED ANALYSIS
# Leave this False to inspect settings and source counts without calculating
# placement rates or writing plots. Charles changes it to True in Cursor.
RUN_ANALYSIS = False

if not RUN_ANALYSIS:
    print("Analysis did not run. Change RUN_ANALYSIS to True, then rerun this cell.")
else:
    result = pilot.run_pilot(settings, run=True)
    run_folder, summary = result
    print("\nSaved this run in:", run_folder)
    print("Read first:", run_folder / "report.md")
    display(summary[[
        "county", "tier", "source_applicants", "primary_denominator",
        "primary_successes", "primary_observed_success_fraction",
        "qualifying_programs", "qualifying_program_places",
    ]].rename(columns={
        "county": "County", "tier": "Top tiers",
        "source_applicants": "Source applicants",
        "primary_denominator": "Outcome denominator",
        "primary_successes": "Selected",
        "primary_observed_success_fraction": "Observed fraction",
        "qualifying_programs": "Winning programs",
        "qualifying_program_places": "Program places",
    }))

## Cell 4: read the story and choose a figure

After Cell 3 runs, this cell opens the generated report and displays **one chosen PNG** inline. Change `TIER_TO_VIEW` between 1 and 2 and `FIGURE_TO_VIEW` among `"hero"`, `"combined_hero"`, `"county_bands"`, and `"pooled_bands"`. The HERO figure has four county rows plus a combined row; `"combined_hero"` displays that combined row alone. Each score-band PNG has **five data panels**: four separate counties plus an applicant-weighted combination of all four. The orange line marks the proposed band of excellence and red marks the top band, using the percentile cuts chosen in Cell 1. Every stronger-band point is labeled with successful placements/applicants and the number of distinct gymnasiums behind it; pooled points also show how many counties contributed.

All bin counts are also saved as CSV files in the run folder. A striking point backed by one gymnasium is a description of that gymnasium, not a replicated peer-environment pattern.

In [ ]:
# CELL 4 — DISPLAY SAVED OUTPUTS ONLY; NO RECOMPUTATION
TIER_TO_VIEW = 1
FIGURE_TO_VIEW = "county_bands"  # "hero", "combined_hero", "county_bands", or "pooled_bands"

if "run_folder" not in globals():
    print("Cell 3 has not produced a run folder in this kernel.")
else:
    display(Markdown((run_folder / "report.md").read_text(encoding="utf-8")))
    filename = {
        "hero": f"hero_top{TIER_TO_VIEW}.png",
        "combined_hero": f"hero_combined_top{TIER_TO_VIEW}.png",
        "county_bands": f"conditional_county_bands_top{TIER_TO_VIEW}.png",
        "pooled_bands": f"conditional_pooled_bands_top{TIER_TO_VIEW}.png",
    }[FIGURE_TO_VIEW]
    figure_path = run_folder / filename
    if not figure_path.exists():
        print("This figure was not requested by the current settings:", figure_path)
    else:
        print("Displaying:", figure_path)
        display(Image(filename=str(figure_path)))

## How to read this pilot

A curve compares **actual placements** among applicants who spent time in different originating gymnasiums and had similar measured exam performance. It does not tell us which programs they applied for, whether their classmates caused their placement, or whether an observed downturn is congestion. The report must keep those limits visible beside any promising pattern.

The notebook intentionally does not delete unresolved source items or classify the one missing Galați outcome. All outputs are saved under `education_analysis/outputs/romania_2001_four_county_descriptive_pilot/`, one timestamped folder per run. The raw Ministry pages remain outside the repository.